# LANGKAH-LANGKAH PRAKTIKUM
## Langkah 1: Persiapan Environment & Pembersihan Dependensi
Buka notebook Google Colab baru, beri nama Praktikum_RAG_GenAI_Permenkes.ipynb, lalu jalankan kode berikut untuk menginstal seluruh dependensi framework RAG yang dibutuhkan (termasuk SDK resmi google-genai):

In [ ]:
# Install official Google GenAI SDK, ChromaDB vector database, and sentence-transformers
!pip install -q google-genai chromadb sentence-transformers langchain-community

# Import main modules
import os
import getpass
import numpy as np
import pandas as pd

# Import official Google GenAI client module
from google import genai
from google.genai import types

# Import text splitter and vector database components
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("Instalasi dan impor pustaka RAG selesai!")

## Langkah 2: Konfigurasi API Key Gemini (Google AI Studio)
SDK terbaru dari Google (google-genai) secara otomatis mencari variabel lingkungan (environment variable) bernama GEMINI_API_KEY untuk melakukan autentikasi ke Google AI Studio.

In [ ]:
# Request API Key securely in Google Colab
if "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Masukkan Google Gemini API Key Anda: ")

print("Koneksi API Key berhasil disiapkan!")

## Langkah 3: Membuat Dataset Dokumen Kustom (Simulasi Permenkes No. 10 Tahun 2024)
Kita akan membuat file dokumen teks lokal berisi poin-poin krusial dari "Peraturan Menteri Kesehatan Republik Indonesia Nomor 10 Tahun 2024 tentang Penyelenggaraan Imunisasi" secara otomatis menggunakan Python sebagai basis pengetahuan sistem RAG:

In [ ]:
# Create custom text data to simulate Permenkes document
permenkes_data = """
PERATURAN MENTERI KESEHATAN REPUBLIK INDONESIA NOMOR 10 TAHUN 2024
TENTANG PENYELENGGARAAN IMUNISASI

Bab I: Ketentuan Umum dan Klasifikasi Imunisasi
Penyelenggaraan Imunisasi bertujuan untuk menurunkan angka kesakitan, kecacatan, dan kematian akibat Penyakit yang Dapat Dicegah Dengan Imunisasi (PD3I). Imunisasi diklasifikasikan menjadi Imunisasi Program dan Imunisasi Pilihan. Imunisasi Program terdiri atas Imunisasi Rutin, Imunisasi Tambahan, dan Imunisasi Khusus.

Bab II: Standar Penyelenggaraan Pelayanan Imunisasi
Pelayanan Imunisasi Program wajib diselenggarakan oleh fasilitas pelayanan kesehatan pemerintah dan dapat melibatkan fasilitas pelayanan kesehatan swasta. Setiap fasilitas pelayanan kesehatan yang menyelenggarakan Imunisasi wajib mencatat dan melaporkan setiap logistik, cakupan, dan kejadian ikutan pasca imunisasi secara berkala ke sistem informasi kesehatan nasional (SatuSehat).

Bab III: Rantai Dingin Penyimpanan Vaksin (Cold Chain)
Penyimpanan vaksin harus mengikuti standar rantai dingin (cold chain) untuk menjaga efektivitas dan kualitas vaksin. Vaksin sensitif beku (seperti Hepatitis B, DPT-HB-Hib, IPV) wajib disimpan pada suhu antara positif 2 derajat Celsius sampai dengan positif 8 derajat Celsius. Vaksin sensitif panas (seperti Polio oral/OPV) harus disimpan pada suhu minus 15 derajat Celsius sampai dengan minus 25 derajat Celsius di tingkat provinsi atau kabupaten.

Bab IV: Kejadian Ikutan Pasca Imunisasi (KIPI)
Setiap fasilitas kesehatan dan tenaga medis wajib melakukan pemantauan, pencatatan, dan pelaporan terhadap kasus Kejadian Ikutan Pasca Imunisasi (KIPI) yang ditemukan. Laporan kasus KIPI serius harus disampaikan paling lambat 24 jam sejak kasus ditemukan kepada Dinas Kesehatan Kabupaten/Kota setempat untuk segera dilakukan investigasi oleh Komite Daerah (Komda) PP-KIPI.
"""

# Save the text into a local file
with open("permenkes_no_10_2024.txt", "w", encoding="utf-8") as f:
    f.write(permenkes_data.strip())

print("File permenkes_no_10_2024.txt berhasil dibuat sebagai basis data kustom RAG!")

## Langkah 4: Pemrosesan Dokumen (Document Chunking)
Dokumen regulasi hukum yang panjang harus dipecah menjadi bagian-bagian kecil (chunks) agar pencarian vektor lebih spesifik dan tidak melebihi batasan ukuran input (context window) dari LLM.

In [ ]:
# Read the custom text document
with open("permenkes_no_10_2024.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

# Initialize recursive character text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,       
    chunk_overlap=50      # Overlapping characters between adjacent chunks
)

# Split text document into chunks
chunks = text_splitter.split_text(text_data)

print(f"Total Chunks yang terbentuk: {len(chunks)}")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk)

## Langkah 5: Pembuatan Embeddings BAAI/bge-m3 & Penyimpanan ke Database Vektor
Kita akan mengonfigurasi model embedding BAAI/bge-m3 melalui Hugging Face. Model ini menghasilkan representasi vektor dengan akurasi semantik tinggi dalam 1024 dimensi spasial.

In [ ]:
# Configure embeddings model BAAI
model_name = "BAAI/bge-m3"

# In Hugging Face Langchain ecosystem device parameter is configured via model kwargs
model_kwargs = {'device': 'cpu'} 

# Initialize BAAI embeddings generator
print(f"Mengunduh dan menyiapkan model embeddings ({model_name}) di device: {model_kwargs['device']}...")
embeddings_model = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs={'normalize_embeddings': True} # Normalize embeddings for accurate cosine similarity
)

# Create Vector Store based on ChromaDB
vector_db = Chroma.from_texts(
    texts=chunks,
    embedding=embeddings_model,
    persist_directory="./chroma_db_permenkes" # Persist database to local folder
)

print("Database Vektor ChromaDB (dengan BAAI/bge-m3) berhasil diinisialisasi!")

## Langkah 6: Membangun Pipeline RAG Manual (Native Retriever & Generator)
Di sini, kita tidak lagi menggunakan komponen black-box seperti RetrievalQA dari LangChain. Kita akan menulis fungsi Python secara manual untuk melakukan Retrieve dari ChromaDB, melakukan Augment pada prompt, dan melakukan Generate menggunakan SDK resmi google-genai dengan model Gemini 3.5 Flash.

In [ ]:
# Initialize Google GenAI client globally
# Client automatically reads the environment variable GEMINI_API_KEY
client = genai.Client()

def rag_query(query_text):
    # Retrieve relevant documents
    # Fetch nearest chunks based on semantic similarity
    docs = vector_db.similarity_search(query_text, k=2)
    context = "\n\n".join([doc.page_content for doc in docs])
    
    # Augment context and construct prompt
    prompt = f"""Gunakan potongan informasi konteks berikut untuk menjawab pertanyaan di akhir secara akurat sesuai peraturan hukum yang berlaku. 
Gunakan hanya informasi dari konteks di bawah ini. Jika Anda tidak mengetahui jawabannya berdasarkan konteks regulasi yang diberikan, katakan secara jujur bahwa Anda tidak tahu. JANGAN mencoba mengarang dasar hukum atau sanksi di luar dokumen ini.

Konteks Regulasi (Permenkes No. 10 Tahun 2024):
{context}

Pertanyaan Pengguna: {query_text}

Jawaban Regulasi Anda (Sampaikan secara formal, terstruktur, dan berbasis hukum):"""
    
    # Generate response using Gemini AI Studio
    # Use low temperature to suppress creativity and prevent hallucination
    config = types.GenerateContentConfig(
        temperature=0.1
    )
    
    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt,
        config=config
    )
    
    return response.text, context

print("Fungsi RAG Chain Native dengan Google GenAI SDK siap!")

## Langkah 7: Pengujian Q&A (Membuktikan Kekuatan RAG Regulasi Kesehatan)
Mari kita uji performa sistem RAG ini dengan mengajukan beberapa pertanyaan medis dan hukum terkait Permenkes No. 10 Tahun 2024 menggunakan fungsi pipeline manual kita.

In [ ]:
# Query regarding immunization classification
query_1 = "Bagaimana pembagian klasifikasi atau jenis Imunisasi Program menurut Permenkes No 10 Tahun 2024?"
print(f"Pertanyaan: {query_1}")

response_1, context_1 = rag_query(query_1)
print("\nJawaban RAG:\n", response_1)

# Query regarding cold chain storage temperature
query_2 = "Berapa suhu standar penyimpanan yang diwajibkan untuk vaksin yang bersifat sensitif beku?"
print(f"Pertanyaan: {query_2}")

response_2, context_2 = rag_query(query_2)
print("\nJawaban RAG:\n", response_2)

# Query testing hallucination limits
query_3 = "Berapa besaran denda finansial maksimal jika puskesmas swasta terlambat melaporkan kasus KIPI?"
print(f"Pertanyaan: {query_3}")

response_3, context_3 = rag_query(query_3)
print("\nJawaban RAG:\n", response_3)